# 03 — Gold: bridge_policy_party

| Property | Value |
|----------|-------|
| **Gold Table** | `bridge_policy_party` |
| **Grain** | One row per PolicyPartyRoleId (Policy × Party × Role) |
| **Source** | `rpt.vwPolicyPartyRole` |
| **PK** | `PolicyPartyRoleId` (int) |
| **Rows** | 48,740,661 |

> 🔴 **Cross-sell engine** — this is the table that links Clients to Policies.
> Filter `GlobalPartyRole = 'Client'` + `IsPrimaryParty = true` for cross-sell analysis.

In [ ]:
# ============================================================
# Cell 1: Setup & Config
# ============================================================
from pyspark.sql import functions as F

spark.conf.set("spark.sql.parquet.datetimeRebaseModeInRead", "CORRECTED")
spark.conf.set("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED")

LAKEHOUSE = "The_Global_Loom"
TABLE = "bridge_policy_party"
SOURCE_TABLE = "rpt.vwPolicyPartyRole"

print(f"✅ Config: {SOURCE_TABLE} → {LAKEHOUSE}.{TABLE}")

In [ ]:
# ============================================================
# Cell 2: Read silver source
# ============================================================
df_src = spark.table(SOURCE_TABLE)

print(f"📥 Source: {df_src.count():,} rows × {len(df_src.columns)} cols")
df_src.printSchema()

## Cell 3: Transform

- Keep: PolicyId (FK), PartyId (FK), GlobalPartyId (cross-sell), GlobalPartyRole, IsPrimaryParty
- Drop: SourceQuery, PartyKey, PolicyKey, PartyRoleKey, PartyRoleId, ETL dates

In [ ]:
# ============================================================
# Cell 3: Transform
# ============================================================
df_clean = df_src.select(
    F.col("PolicyPartyRoleId").cast("int"),
    F.col("PolicyId").cast("int"),
    F.col("PartyId").cast("int"),
    F.col("GlobalPartyId").cast("int"),
    F.col("GlobalPartyRoleId").cast("int"),
    F.col("GlobalPartyRole").cast("string"),
    F.col("IsPrimaryParty").cast("boolean"),
    F.col("DataSourceInstanceId").cast("int"),
    F.col("IsDeleted").cast("boolean")
)

print(f"✅ After column select: {df_clean.count():,} rows × {len(df_clean.columns)} cols")
print(f"   Dropped {len(df_src.columns) - len(df_clean.columns)} columns")

In [ ]:
# ============================================================
# Cell 4: Data quality checks
# ============================================================
total = df_clean.count()
dupes = total - df_clean.select("PolicyPartyRoleId").distinct().count()
null_policy = df_clean.filter(F.col("PolicyId").isNull()).count()
null_party = df_clean.filter(F.col("PartyId").isNull()).count()

# Cross-sell stats
client_rows = df_clean.filter(F.col("GlobalPartyRole") == "Client").count()
primary_client_rows = df_clean.filter(
    (F.col("GlobalPartyRole") == "Client") & (F.col("IsPrimaryParty") == True)
).count()

print(f"✅ DQ Checks")
print(f"   Total rows:          {total:,}")
print(f"   Duplicate PKs:       {dupes}")
print(f"   Null PolicyIds:      {null_policy}")
print(f"   Null PartyIds:       {null_party}")
print(f"   Client rows:         {client_rows:,}")
print(f"   Primary client rows: {primary_client_rows:,}")

assert dupes == 0, f"❌ Duplicates!"
print("\n✅ All DQ checks passed")

In [ ]:
# ============================================================
# Cell 5: Write to gold lakehouse
# ============================================================
df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE)

print(f"✅ Written: {TABLE}")
print(f"   Rows: {spark.table(TABLE).count():,}")